# Notebook 02: Automate model governance with EventBridge and Model Cards

The MLflow sync creates a draft Model Card for each version. Here an EventBridge-triggered Lambda completes it with governance and evaluation details and keeps its status in step with approval.

|||
|---|---|
|00. Environment setup ||
|01. Experiment in a notebook, then scale with SageMaker AI jobs ||
|**02. Automate model governance with EventBridge and Model Cards** |**◄ YOU ARE HERE**|
|03. Operationalize with an ML pipeline and model registry ||
|04. Deploy to an endpoint and monitor ||
|05. Templatize staged deployment as a Project ||

## What you do here

1. Deploy the Lambda function [`model_card_automation/lambda_function.py`](model_card_automation/lambda_function.py).
2. Create an EventBridge rule on `SageMaker Model Package State Change` for the workshop's groups.
3. Backfill the notebook 01 version and inspect the result.

**Outcome:** every registered version carries a completed Model Card whose status tracks the approval workflow, with no manual step.

![EventBridge invokes Lambda to enrich the Model Card on the same SageMaker Model Package version](img/model-card-automation-aws.svg)

<div class="alert alert-info">
This uses the card that every Model Package version carries (<code>ModelCard</code> in <code>DescribeModelPackage</code>), not a standalone <code>CreateModelCard</code> resource. <a href="https://aws.amazon.com/blogs/machine-learning/improve-governance-of-models-with-amazon-sagemaker-unified-model-cards-and-model-registry/">More on unified Model Cards</a>.
</div>

## Setup

In [ ]:
import sys
assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

In [ ]:
%store -r 

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

In [ ]:
import io
import json
import time
import zipfile
from pathlib import Path
from time import gmtime, strftime

import boto3
from botocore.exceptions import ClientError

# V3: Core imports
from sagemaker.core.helper.session_helper import Session, get_execution_role
from IPython.display import HTML, display

session = Session()
sm_role = get_execution_role()
region = session.boto_region_name
account_id = session.account_id()

sm_client = boto3.client("sagemaker")
iam_client = boto3.client("iam")
lambda_client = boto3.client("lambda")
events_client = boto3.client("events")

## Set constants

In [ ]:
project = "aim408-from-idea-to-prod"

# Reuse the MLflow registered model created in notebook 01 and restored via %store.
# Fall back to the same date-suffixed (-DDMMYY) name when running this standalone.
date_suffix = strftime('%d%m%y', gmtime())
if "registered_model_name" not in dir():
    registered_model_name = f"aim408-from-idea-to-prod-model-{date_suffix}"
    %store registered_model_name

# NOT the same string as the MLflow registered model: the auto-sync appends a short
# random suffix when it creates the SageMaker Model Package Group. Resolved for real
# against the registry in the backfill cell below; this is only a starting guess.
model_package_group_name = registered_model_name

# The EventBridge rule matches on this *prefix*, so it also covers the model package
# groups that other runs of the workshop create on other days. Notebook 04 registers
# nothing: it deploys the version notebook 03 registers into this same group.
model_package_group_prefix = f"{project}-model-"

model_card_lambda_name = f"{project}-model-card-automation"
model_card_lambda_role_name = f"{project}-model-card-automation-role"
model_card_rule_name = f"{project}-model-card-on-registration"

%store model_card_lambda_name
%store model_card_rule_name

print(f"MLflow registered model:  {registered_model_name}")
print(f"Rule matches groups:      {model_package_group_prefix}*")
print(f"Lambda function:          {model_card_lambda_name}")
print(f"EventBridge rule:         {model_card_rule_name}")

<details>
<summary><b>Permissions</b></summary>

The workshop stack grants the Studio role what this notebook needs (Lambda, EventBridge, `iam:PassRole`, `logs:FilterLogEvents`) through `ModelCardAutomationAccessPolicy`, scoped to the function and rule names above. It also creates the Lambda execution role, because the Studio role has no `iam:CreateRole`. Outside the workshop, set `LAMBDA_ROLE_ARN` to a role that trusts `lambda.amazonaws.com` and allows `sagemaker:DescribeModelPackage`, `sagemaker:UpdateModelPackage`, and CloudWatch Logs.
</details>

## What the function does

| Model Card rule | Function behavior |
|---|---|
| One card per package version | Writes the card on the version with `UpdateModelPackage(ModelCard=...)` |
| Content changes create a new card version | Enriches content **once**, then only updates status |
| Status workflow | `PendingManualApproval` → `PendingReview`, `Approved` → `Approved`, `Rejected` → `Archived` |
| Approved cards are frozen | For an already-approved package (notebook 01), writes as `PendingReview`, then approves |
| Keep source details | Keeps the sync's training metrics; adds evaluation results from `CustomerMetadataProperties` |

SageMaker can deliver several events per change, and the function's own update emits one more, so it is **idempotent**: it re-reads the package and skips the write when nothing changed.

The code is plain `boto3`; the cell lists its functions.

In [ ]:
lambda_source_path = Path("model_card_automation/lambda_function.py")
lambda_source = lambda_source_path.read_text()

# Print the module docstring and the function signatures — open the file for the rest.
import ast
module = ast.parse(lambda_source)
print(ast.get_docstring(module))
print("\nFunctions:")
for node in module.body:
    if isinstance(node, ast.FunctionDef):
        print(f"  {node.name}({', '.join(a.arg for a in node.args.args)})")

## Look up the Lambda execution role

The role is pre-created by the workshop stack with only the permissions the function needs. The cell prints its policy.

In [ ]:
# Override only if you run this notebook outside the workshop account and want to
# point the function at a role of your own. In the workshop, leave it at None.
LAMBDA_ROLE_ARN = None

if LAMBDA_ROLE_ARN is None:
    try:
        model_card_lambda_role_arn = iam_client.get_role(RoleName=model_card_lambda_role_name)["Role"]["Arn"]
        print(f"Using the pre-created role: {model_card_lambda_role_arn}")
    except iam_client.exceptions.NoSuchEntityException:
        raise RuntimeError(
            f"IAM role '{model_card_lambda_role_name}' does not exist.\n"
            "It is provisioned by the workshop CloudFormation stack as the resource\n"
            "ModelCardAutomationLambdaRole in static/cfnSageMakerProjectsRoles.yaml.\n"
            "Update that stack, or set LAMBDA_ROLE_ARN above to a role that trusts\n"
            "lambda.amazonaws.com and allows sagemaker:DescribeModelPackage,\n"
            "sagemaker:UpdateModelPackage and CloudWatch Logs."
        ) from None
else:
    model_card_lambda_role_arn = LAMBDA_ROLE_ARN
    print(f"Using provided role: {model_card_lambda_role_arn}")

# Print the least-privilege policy that is actually deployed on the role, so the
# blast radius of the function is visible before you create it.
role_name = model_card_lambda_role_arn.split("/")[-1]
try:
    for policy_name in iam_client.list_role_policies(RoleName=role_name)["PolicyNames"]:
        document = iam_client.get_role_policy(RoleName=role_name, PolicyName=policy_name)["PolicyDocument"]
        print(f"\nInline policy '{policy_name}':")
        for statement in document["Statement"]:
            actions = statement["Action"] if isinstance(statement["Action"], list) else [statement["Action"]]
            resources = statement["Resource"] if isinstance(statement["Resource"], list) else [statement["Resource"]]
            print(f"  {statement.get('Sid', '(no sid)'):<20}{statement['Effect']} {', '.join(actions)}")
            for resource in resources:
                print(f"  {'':<20}  on {resource}")
except ClientError as e:
    print(f"\n[WARN] Could not read the inline policies of {role_name} ({e.response['Error']['Code']}).")

%store model_card_lambda_role_arn

## Deploy the Lambda function

The governance text (purpose, risk rating, stakeholders) is passed as environment variables. Re-running the cell updates the function.

In [ ]:
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(lambda_source_path, arcname="lambda_function.py")
zip_bytes = buffer.getvalue()

lambda_environment = {
    "Variables": {
        "MODEL_CARD_CREATOR": f"{project} workshop ({user_profile_name if 'user_profile_name' in dir() else 'sagemaker-user'})",
        "MODEL_CARD_PURPOSE": "Predict whether a bank customer will subscribe to a term deposit after a direct marketing call.",
        "MODEL_CARD_INTENDED_USES": "Rank customers for outbound marketing campaigns. Not intended for credit or eligibility decisions.",
        "MODEL_CARD_RISK_RATING": "Low",
        "MODEL_CARD_BUSINESS_PROBLEM": "Increase the conversion rate of term-deposit campaigns while reducing call volume.",
        "MODEL_CARD_STAKEHOLDERS": "Retail banking marketing team",
        "MODEL_CARD_LINE_OF_BUSINESS": "Retail banking",
    }
}

try:
    lambda_client.get_function(FunctionName=model_card_lambda_name)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=model_card_lambda_name)
    lambda_client.update_function_code(FunctionName=model_card_lambda_name, ZipFile=zip_bytes)
    lambda_client.get_waiter("function_updated_v2").wait(FunctionName=model_card_lambda_name)
    lambda_client.update_function_configuration(
        FunctionName=model_card_lambda_name, Role=model_card_lambda_role_arn,
        Environment=lambda_environment, Timeout=60,
    )
    print(f"Updated function: {model_card_lambda_name}")
except lambda_client.exceptions.ResourceNotFoundException:
    for attempt in range(6):
        try:
            lambda_client.create_function(
                FunctionName=model_card_lambda_name,
                Runtime="python3.12",
                Role=model_card_lambda_role_arn,
                Handler="lambda_function.lambda_handler",
                Code={"ZipFile": zip_bytes},
                Description="Completes the Model Card of every model package version registered in the aim408-from-idea-to-prod groups",
                Timeout=60,
                MemorySize=256,
                Environment=lambda_environment,
                Tags={"project": project},
            )
            print(f"Created function: {model_card_lambda_name}")
            break
        except ClientError as e:
            # "The role defined for the function cannot be assumed by Lambda" = IAM propagation delay
            if e.response["Error"]["Code"] == "InvalidParameterValueException" and "assumed" in str(e) and attempt < 5:
                print("Role not yet assumable, retrying in 10s...")
                time.sleep(10)
            else:
                raise

lambda_client.get_waiter("function_active_v2").wait(FunctionName=model_card_lambda_name)
model_card_lambda_arn = lambda_client.get_function(FunctionName=model_card_lambda_name)["Configuration"]["FunctionArn"]
print(f"Function ARN: {model_card_lambda_arn}")

## Create the EventBridge rule

The rule matches [`SageMaker Model Package State Change`](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html#eventbridge-model-package) events whose group name starts with the workshop prefix, so it covers every day's group without touching others.

In [ ]:
event_pattern = {
    "source": ["aws.sagemaker"],
    "detail-type": ["SageMaker Model Package State Change"],
    "detail": {
        "ModelPackageGroupName": [{"prefix": model_package_group_prefix}],
    },
}

rule_arn = events_client.put_rule(
    Name=model_card_rule_name,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Complete the Model Card of every aim408-from-idea-to-prod model package version",
    Tags=[{"Key": "project", "Value": project}],
)["RuleArn"]
print(f"Rule: {rule_arn}")

events_client.put_targets(
    Rule=model_card_rule_name,
    Targets=[{"Id": "model-card-automation", "Arn": model_card_lambda_arn}],
)

try:
    lambda_client.add_permission(
        FunctionName=model_card_lambda_name,
        StatementId="allow-eventbridge-model-card-rule",
        Action="lambda:InvokeFunction",
        Principal="events.amazonaws.com",
        SourceArn=rule_arn,
    )
    print("EventBridge is allowed to invoke the function.")
except lambda_client.exceptions.ResourceConflictException:
    print("Invoke permission already present.")

print(json.dumps(event_pattern, indent=2))

## Backfill the notebook 01 version

The rule only sees future events, so invoke the function directly with the same event shape for every existing version. The notebook 01 version is already `Approved`, so expect the content to be written as `PendingReview` and then approved.

<div class="alert alert-warning">
The sync appends a random suffix to the group name (<code>...-model-130926</code> → <code>...-model-130926-9cd7c1</code>), so the cell searches the registry for the group.
</div>

In [ ]:
def model_package_event(summary):
    # Same shape as the EventBridge "SageMaker Model Package State Change" event
    return {
        "version": "0",
        "id": "manual-backfill",
        "detail-type": "SageMaker Model Package State Change",
        "source": "aws.sagemaker",
        "account": account_id,
        "region": region,
        "resources": [summary["ModelPackageArn"]],
        "detail": {
            "ModelPackageGroupName": model_package_group_name,
            "ModelPackageVersion": summary["ModelPackageVersion"],
            "ModelPackageArn": summary["ModelPackageArn"],
            "ModelApprovalStatus": summary.get("ModelApprovalStatus"),
        },
    }

def versions_in_group(group_name):
    """Versions of a model package group, oldest first.

    Returns [] for a group that does not exist: ListModelPackages does not raise on an
    unknown group name, so "group is empty" and "no such group" look identical here.
    """
    summaries = []
    paginator = sm_client.get_paginator("list_model_packages")
    for page in paginator.paginate(ModelPackageGroupName=group_name, SortBy="CreationTime", SortOrder="Ascending"):
        summaries.extend(page["ModelPackageSummaryList"])
    return summaries


def groups_containing(fragment):
    """Model package group names containing `fragment`, most recently created first."""
    # NameContains must match ^[a-zA-Z0-9](-*[a-zA-Z0-9]){0,62}$, so a trailing hyphen
    # (which model_package_group_prefix has) is rejected -- strip it before the call.
    names = []
    paginator = sm_client.get_paginator("list_model_package_groups")
    for page in paginator.paginate(
        NameContains=fragment.rstrip("-"), SortBy="CreationTime", SortOrder="Descending"
    ):
        names.extend(g["ModelPackageGroupName"] for g in page["ModelPackageGroupSummaryList"])
    return names


# Search narrowest first: the MLflow name as-is, then the groups the sync derived from
# it, then any group of this workshop (so an earlier day's run is still picked up).
search_order = [registered_model_name]
for name in groups_containing(registered_model_name) + [
    g for g in groups_containing(model_package_group_prefix) if g.startswith(model_package_group_prefix)
]:
    if name not in search_order:
        search_order.append(name)

packages = []
for candidate in search_order:
    packages = versions_in_group(candidate)
    if packages:
        model_package_group_name = candidate
        break

if not packages:
    raise RuntimeError(
        f"No model package version in any '{model_package_group_prefix}*' group in {region}.\n"
        f"Groups checked: {search_order}\n"
        "Run 01-mlflow-and-sagemaker-jobs.ipynb through the 'Register, and watch the "
        "auto-sync'\ncell in Part 2, then re-run this notebook from the top."
    )

if model_package_group_name != registered_model_name:
    print(f"MLflow registered model: {registered_model_name}")
print(f"Model Package Group:     {model_package_group_name} ({len(packages)} version(s))\n")

for summary in packages:
    test_event = model_package_event(summary)
    response = lambda_client.invoke(
        FunctionName=model_card_lambda_name,
        InvocationType="RequestResponse",
        Payload=json.dumps(test_event).encode(),
    )
    result = json.loads(response["Payload"].read())
    if response.get("FunctionError"):
        raise RuntimeError(f"Lambda failed for {summary['ModelPackageArn']}: {result}")
    print(f"v{summary['ModelPackageVersion']:<3} {summary.get('ModelApprovalStatus', '(no approval status)'):<24} -> {result['action']}")

# The most recent version is inspected in the next cells
model_package_arn = packages[-1]["ModelPackageArn"]
model_package_version = packages[-1]["ModelPackageVersion"]

### Inspect the completed card

Training metrics come from the sync; everything else from the function.

In [ ]:
described = sm_client.describe_model_package(ModelPackageName=model_package_arn)
card = json.loads(described["ModelCard"]["ModelCardContent"])

print(f"Model package:   {model_package_group_name} v{model_package_version}")
print(f"Approval status: {described.get('ModelApprovalStatus', '(not set)')}")
print(f"Card status:     {described['ModelCard']['ModelCardStatus']}")

def section(title, obj):
    print(f"\n{title}")
    for k, v in obj.items():
        print(f"  {k:<36}{v if not isinstance(v, (dict, list)) else json.dumps(v)[:100]}")

section("Model overview", card.get("model_overview", {}))
section("Intended uses", card.get("intended_uses", {}))
section("Business details", card.get("business_details", {}))
for evaluation in card.get("evaluation_details", []):
    print(f"\nEvaluation: {evaluation['name']}")
    for group in evaluation.get("metric_groups", []):
        for m in group["metric_data"]:
            print(f"  {m['name']:<36}{m['value']}")
training = card.get("training_details", {}).get("training_job_details", {})
print("\nTraining metrics (from the MLflow sync)")
for m in training.get("training_metrics", []):
    print(f"  {m['name']:<36}{m['value']:.4f}")
section("Custom details", card.get("additional_information", {}).get("custom_details", {}))

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions/version-{}">the model version</a> in the Studio Model Registry UI (Overview tab shows the model card, Audit shows its status)</b>'.format(
        domain_id, region, model_package_group_name, model_package_version
    ))
)

### Check idempotency

Send the same event again: no new card version.

In [ ]:
response = lambda_client.invoke(
    FunctionName=model_card_lambda_name,
    InvocationType="RequestResponse",
    Payload=json.dumps(test_event).encode(),
)
print(json.loads(response["Payload"].read())["action"])

### Watch real events (optional)

The function's own updates already produced real events; the logs show them ending in `no-op`.

In [ ]:
logs_client = boto3.client("logs")
log_group = f"/aws/lambda/{model_card_lambda_name}"
try:
    time.sleep(15)  # give EventBridge + Lambda + CloudWatch a moment
    events = logs_client.filter_log_events(
        logGroupName=log_group,
        startTime=int((time.time() - 15 * 60) * 1000),
        filterPattern='"model-package/"',
    )["events"]
    for e in events[-10:]:
        print(e["message"].strip()[-160:])
    if not events:
        print("No log lines yet; re-run this cell in a minute.")
except Exception as e:
    print(f"[WARN] Could not read {log_group} ({e}). Open it in the CloudWatch console instead.")

## Every version, one card

Lists each version with its approval and card status. Come back after notebook 03 to see the pipeline version documented automatically.

In [ ]:
def model_card_overview(group_name):
    rows = []
    paginator = sm_client.get_paginator("list_model_packages")
    for page in paginator.paginate(ModelPackageGroupName=group_name, SortBy="CreationTime", SortOrder="Ascending"):
        for summary in page["ModelPackageSummaryList"]:
            d = sm_client.describe_model_package(ModelPackageName=summary["ModelPackageArn"])
            mc = d.get("ModelCard") or {}
            content = json.loads(mc["ModelCardContent"]) if mc.get("ModelCardContent") else {}
            custom = content.get("additional_information", {}).get("custom_details", {})
            rows.append({
                "version": d["ModelPackageVersion"],
                "approval": d.get("ModelApprovalStatus", "(not set)"),
                "card_status": mc.get("ModelCardStatus", "(no card)"),
                "automated": "yes" if custom.get("ModelCardAutomation") else "no",
                "risk": content.get("intended_uses", {}).get("risk_rating", ""),
                "created": d["CreationTime"].strftime("%Y-%m-%d %H:%M"),
            })
    return rows

print(f"{'ver':<5}{'approval':<24}{'card status':<15}{'automated':<11}{'risk':<8}created")
for r in model_card_overview(model_package_group_name):
    print(f"{r['version']:<5}{r['approval']:<24}{r['card_status']:<15}{r['automated']:<11}{r['risk']:<8}{r['created']}")

## Clean-up

**Keep the automation** for notebooks 03–05; it costs nothing while idle. At the end of the workshop, set `DELETE_AUTOMATION = True`. The Lambda role belongs to the workshop stack.

In [ ]:
DELETE_AUTOMATION = False

if DELETE_AUTOMATION:
    try:
        events_client.remove_targets(Rule=model_card_rule_name, Ids=["model-card-automation"])
        events_client.delete_rule(Name=model_card_rule_name)
        print(f"Deleted rule {model_card_rule_name}")
    except ClientError as e:
        print(f"[WARN] rule: {e}")
    try:
        lambda_client.delete_function(FunctionName=model_card_lambda_name)
        print(f"Deleted function {model_card_lambda_name}")
    except ClientError as e:
        print(f"[WARN] function: {e}")
    # The Lambda execution role is not deleted here: it belongs to the workshop
    # CloudFormation stack and goes away when that stack is deleted.
    print(f"Left role {model_card_lambda_role_name} in place (owned by CloudFormation).")
else:
    print("Automation kept (DELETE_AUTOMATION=False).")

## Summary

- Each Model Package version carries one card, immutable except for its status.
- A Lambda function, triggered by EventBridge, completes the card and aligns its status with approval.
- Repeated events are idempotent.

## Continue with notebook 03

Open [notebook 03](03-sagemaker-pipeline.ipynb). It checks the card of the pipeline-registered version at the end.

## Further development ideas for your real-world projects

- **Wire the approval gate to deployment.** Notebooks 00, 01 and 03 already separate the personas: the data-scientist role can register and stage, only the governance-officer role can set `ModelApprovalStatus=Approved`. The automation here documents, the officer approves, and the resulting `Approved` event (`detail.ModelApprovalStatus`) is the natural trigger for a deployment pipeline
- Add a second target to the rule: an **SNS topic** or a **Step Functions** approval workflow that notifies reviewers when a card enters `PendingReview`
- Store the governance text (purpose, risk rating, stakeholders) per model package group as **group tags** and have the function read them, instead of function environment variables
- Add the **SHAP feature importances and fairness metrics** produced by the `explain` step to `evaluation_details`: have `register.py` pass them through `CustomerMetadataProperties` (or an S3 JSON referenced from the card) so the function can render them next to the test AUC
- Export approved cards to PDF with `CreateModelCardExportJob` for audit archives

## Additional resources

- [Amazon SageMaker Model Cards](https://docs.aws.amazon.com/sagemaker/latest/dg/model-cards.html)
- [Model card FAQs — versioning and Model Registry integration](https://docs.aws.amazon.com/sagemaker/latest/dg/model-cards-faqs.html)
- [Update the details of a model version (model package model card schema)](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-details.html)
- [Improve governance of models with unified Model Cards and Model Registry](https://aws.amazon.com/blogs/machine-learning/improve-governance-of-models-with-amazon-sagemaker-unified-model-cards-and-model-registry/)
- [Automating Amazon SageMaker AI with Amazon EventBridge](https://docs.aws.amazon.com/sagemaker/latest/dg/automating-sagemaker-with-eventbridge.html)
- [EventBridge event patterns: prefix matching](https://docs.aws.amazon.com/eventbridge/latest/userguide/eb-event-patterns-content-based-filtering.html)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>